# 🎬 Aiscern — Video Deepfake Detector
### Google Colab T4 | ViT-base + LoRA | Full 155k face crops × 8 epochs

**Platform**: Google Colab (T4 12GB) — run in 2 sessions  
**Session 1**: ~12h → trains epochs 1–7, saves checkpoint  
**Session 2**: ~1.5h → resumes, finishes epoch 8, pushes model  
**Output model**: `saghi776/aiscern-video-detector`  
**Expected accuracy**: 88% → **95%**

### Strategy: Frame crops instead of raw video
Video deepfakes manipulate faces — so we train on extracted face crops.
No video decoding needed → fits T4 12GB comfortably.

### Datasets used
| Dataset | Size | Type |
|---|---|---|
| Celeb-DF v2 | 6,229 | Celebrity deepfake faces |
| FaceForensics++ | 9,000 | Face swap / reenactment |
| DeepFake-vs-Real-Faces | 140,000 | Large-scale face crops |

> ⚠️ **Before starting Session 2**: File → Save a copy in Drive, 
> then mount Drive to restore checkpoint automatically.


In [ ]:
# ── STEP 0: Mount Google Drive (keeps checkpoints across sessions) ─
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
DRIVE_DIR      = '/content/drive/MyDrive/aiscern_video'
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Checkpoints will persist at: {CHECKPOINT_DIR}")


In [ ]:
# ── INSTALL DEPS ─────────────────────────────────────────────────
!pip install -q transformers==4.40.0 datasets peft==0.10.0 accelerate              evaluate scikit-learn Pillow huggingface_hub

import warnings, logging, torch
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu  = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu}  ({mem:.1f} GB VRAM)")
else:
    print("⚠️  No GPU — Runtime → Change runtime type → T4 GPU")


In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────
HF_TOKEN        = 'YOUR_HF_TOKEN_HERE'   # ← paste your HF token

BASE_MODEL      = 'google/vit-base-patch16-224'
PUSH_REPO       = 'saghi776/aiscern-video-detector'
IMG_SIZE        = 224
SAMPLES_PER_CLASS = 77614    # all available (155,229 ÷ 2, rounded)
# T4 12GB: batch=16 for ViT-base (batch=32 would OOM on T4)
BATCH_SIZE      = 16
EPOCHS          = 8
LR              = 2e-4
WARMUP_RATIO    = 0.1
WEIGHT_DECAY    = 0.01
SEED            = 42

import os
os.environ['HF_TOKEN'] = HF_TOKEN
print(f"✅ Config set — {SAMPLES_PER_CLASS*2:,} samples × {EPOCHS} epochs")
print(f"   Batch size: {BATCH_SIZE} (T4 12GB safe)")
print(f"   Session 1 (~12h): epochs 1-7, then Colab disconnects")
print(f"   Session 2 (~1.5h): auto-resumes from checkpoint, finishes epoch 8")


In [ ]:
# ── LOAD DATASETS ────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets
import numpy as np

def normalise_label(val):
    s = str(val).upper().strip()
    if s in ('FAKE','AI','1','GENERATED','DEEPFAKE','TRUE'): return 1
    if s in ('REAL','HUMAN','0','AUTHENTIC','FALSE'):         return 0
    return -1

print("Loading datasets...")
all_splits = []

# 1. DeepFake-vs-Real-Faces — largest, most diverse (140k)
try:
    ds = load_dataset(
        'arnabdhar/DeepFake-Vs-Real-Faces',
        split='train', token=HF_TOKEN
    )
    img_col = next((c for c in ds.column_names if c in ('image','img','Image','face')), None)
    lbl_col = next((c for c in ds.column_names if 'label' in c.lower()), None)
    if img_col and lbl_col:
        if img_col != 'image': ds = ds.rename_column(img_col, 'image')
        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds = ds.filter(lambda x: x['label'] != -1)
        ds = ds.select_columns(['image','label'])
        all_splits.append(ds)
        r = ds.filter(lambda x: x['label']==0).num_rows
        f = ds.filter(lambda x: x['label']==1).num_rows
        print(f"  ✅ DeepFake-vs-Real: {len(ds):,}  (real={r:,} fake={f:,})")
except Exception as e:
    print(f"  ⚠️  DeepFake-vs-Real skipped: {e}")

# 2. Celeb-DF v2 face crops
try:
    ds = load_dataset('haywhy/celeb-df-v2', split='train', token=HF_TOKEN)
    img_col = next((c for c in ds.column_names if c in ('image','img','Image','frame')), None)
    lbl_col = next((c for c in ds.column_names if 'label' in c.lower()), None)
    if img_col and lbl_col:
        if img_col != 'image': ds = ds.rename_column(img_col, 'image')
        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds = ds.filter(lambda x: x['label'] != -1)
        ds = ds.select_columns(['image','label'])
        all_splits.append(ds)
        r = ds.filter(lambda x: x['label']==0).num_rows
        f = ds.filter(lambda x: x['label']==1).num_rows
        print(f"  ✅ Celeb-DF v2: {len(ds):,}  (real={r:,} fake={f:,})")
except Exception as e:
    print(f"  ⚠️  Celeb-DF skipped: {e}")

# 3. FaceForensics++ face swaps
try:
    ds = load_dataset(
        'marcelomoreno26/deepfake-detection',
        split='train', token=HF_TOKEN
    )
    img_col = next((c for c in ds.column_names if c in ('image','img','Image')), None)
    lbl_col = next((c for c in ds.column_names if 'label' in c.lower()), None)
    if img_col and lbl_col:
        if img_col != 'image': ds = ds.rename_column(img_col, 'image')
        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds = ds.filter(lambda x: x['label'] != -1)
        ds = ds.select_columns(['image','label'])
        all_splits.append(ds)
        r = ds.filter(lambda x: x['label']==0).num_rows
        f = ds.filter(lambda x: x['label']==1).num_rows
        print(f"  ✅ FaceForensics++: {len(ds):,}  (real={r:,} fake={f:,})")
except Exception as e:
    print(f"  ⚠️  FaceForensics++ skipped: {e}")

# ── Balance ──────────────────────────────────────────────────────
if not all_splits:
    raise RuntimeError("No datasets loaded — check HF_TOKEN and enable GPU+internet in Colab")

combined = concatenate_datasets(all_splits)
real = combined.filter(lambda x: x['label'] == 0).shuffle(SEED)
fake = combined.filter(lambda x: x['label'] == 1).shuffle(SEED)
n    = min(len(real), len(fake), SAMPLES_PER_CLASS)
balanced = concatenate_datasets([
    real.select(range(n)), fake.select(range(n))
]).shuffle(SEED)

split    = balanced.train_test_split(test_size=0.1, seed=SEED)
train_ds = split['train']
eval_ds  = split['test']
print(f"\n✅ Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}")


In [ ]:
# ── FEATURE EXTRACTION ──────────────────────────────────────────
from transformers import ViTImageProcessor
from PIL import Image as PILImage, ImageEnhance
import numpy as np, random

processor = ViTImageProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)

def safe_img(img):
    if isinstance(img, PILImage.Image):
        return img.convert('RGB').resize((IMG_SIZE, IMG_SIZE), PILImage.LANCZOS)
    return PILImage.fromarray(np.uint8(img)).convert('RGB').resize(
        (IMG_SIZE, IMG_SIZE), PILImage.LANCZOS)

def augment_train(img):
    img = safe_img(img)
    if random.random() > 0.5:
        img = img.transpose(PILImage.FLIP_LEFT_RIGHT)
    if random.random() > 0.5:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.15))
    if random.random() > 0.5:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.15))
    if random.random() > 0.5:
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.8, 1.2))
    return img

def preprocess_train(batch):
    imgs = [augment_train(img) for img in batch['image']]
    batch['pixel_values'] = processor(images=imgs, return_tensors='np')['pixel_values']
    return batch

def preprocess_eval(batch):
    imgs = [safe_img(img) for img in batch['image']]
    batch['pixel_values'] = processor(images=imgs, return_tensors='np')['pixel_values']
    return batch

# Use smaller batch_size for map to avoid T4 OOM during preprocessing
print("Extracting features...")
train_ds = train_ds.map(preprocess_train, batched=True, batch_size=32,
                         remove_columns=['image'], desc='Train')
eval_ds  = eval_ds.map(preprocess_eval,   batched=True, batch_size=32,
                         remove_columns=['image'], desc='Eval')
train_ds.set_format('torch')
eval_ds.set_format('torch')
print(f"✅ Features ready")


In [ ]:
# ── MODEL + LoRA ─────────────────────────────────────────────────
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model
import torch

model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'real', 1: 'fake'},
    label2id={'real': 0, 'fake': 1},
    ignore_mismatched_sizes=True,
    token=HF_TOKEN,
)

# LoRA — trains ~0.5% of params, fits T4 12GB
lora_cfg = LoraConfig(
    r=8,               # slightly lower rank for T4 memory headroom
    lora_alpha=16,
    lora_dropout=0.1,
    bias='none',
    target_modules=['query', 'value', 'key', 'dense'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
model = model.to(device)

# Verify T4 memory won't OOM
if device == 'cuda':
    torch.cuda.empty_cache()
    free_mem = (torch.cuda.get_device_properties(0).total_memory 
                - torch.cuda.memory_allocated()) / 1e9
    print(f"✅ Model on {device} | Free VRAM: {free_mem:.1f} GB")


In [ ]:
# ── TRAIN (with checkpoint auto-resume for Colab sessions) ────────
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate, numpy as np
import glob, os

acc = evaluate.load('accuracy')
f1  = evaluate.load('f1')

def compute_metrics(ep):
    preds = np.argmax(ep.predictions, axis=-1)
    return {
        'accuracy': acc.compute(predictions=preds, references=ep.label_ids)['accuracy'],
        'f1':       f1.compute(predictions=preds, references=ep.label_ids, average='binary')['f1'],
    }

# Find latest checkpoint (auto-resume after Colab disconnects)
checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint-*')))
last_ckpt   = checkpoints[-1] if checkpoints else None
if last_ckpt:
    print(f"🔄 Resuming from: {last_ckpt}")
else:
    print("▶  Starting fresh training")

training_args = TrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,    # 16 for T4 12GB
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 3,             # keep last 3 checkpoints
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    push_to_hub                 = True,
    hub_model_id                = PUSH_REPO,
    hub_token                   = HF_TOKEN,
    hub_strategy                = 'every_save',  # push each checkpoint
    fp16                        = (device == 'cuda'),
    gradient_accumulation_steps = 2,             # effective batch = 32
    dataloader_num_workers      = 2,             # lower for Colab stability
    report_to                   = 'none',
    logging_steps               = 100,
    seed                        = SEED,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training: {len(train_ds):,} samples × {EPOCHS} epochs")
print(f"Effective batch size: {BATCH_SIZE * 2} (gradient accumulation × 2)")
print(f"Session 1: trains to epoch ~7, saves to Drive, Colab may disconnect")
print(f"Session 2: run this notebook again → auto-resumes from epoch 7")
print()

trainer.train(resume_from_checkpoint=last_ckpt)
print("\n✅ Training complete!")


In [ ]:
# ── EVALUATE + PUSH ──────────────────────────────────────────────
results = trainer.evaluate()

print("\n" + "═"*50)
print("FINAL RESULTS")
print("═"*50)
print(f"  Accuracy: {results['eval_accuracy']:.4f}  ({results['eval_accuracy']*100:.2f}%)")
print(f"  F1 Score: {results['eval_f1']:.4f}")
print("═"*50)

trainer.push_to_hub(commit_message=f"Video deepfake detector — acc={results['eval_accuracy']:.4f}")
processor.push_to_hub(PUSH_REPO, token=HF_TOKEN)

from huggingface_hub import HfApi
card = f"""---
license: apache-2.0
tags:
- image-classification
- video-deepfake-detection
- vit
- lora
- face-detection
datasets:
- arnabdhar/DeepFake-Vs-Real-Faces
- haywhy/celeb-df-v2
- marcelomoreno26/deepfake-detection
metrics:
- accuracy
---

# Aiscern Video Deepfake Detector

Fine-tuned `google/vit-base-patch16-224` with LoRA on video deepfake face crops.

**Accuracy**: {results['eval_accuracy']*100:.2f}%  
**F1**: {results['eval_f1']:.4f}  
**Training data**: {len(train_ds):,} face crops from Celeb-DF + FaceForensics++ + DeepFake-vs-Real

## How it works
Extracts face crops from video frames → classifies each as real or fake →
aggregates per-frame predictions for a final video verdict.

## Usage
```python
from transformers import pipeline
detector = pipeline('image-classification', model='{PUSH_REPO}')
# Feed face crop from video frame:
result = detector('frame_crop.jpg')
# [{{'label': 'fake', 'score': 0.95}}]
```
"""

HfApi().upload_file(
    path_or_fileobj=card.encode(), path_in_repo='README.md',
    repo_id=PUSH_REPO, token=HF_TOKEN,
)
print(f"\n✅ Model live: https://huggingface.co/{PUSH_REPO}")
print(f"   Integrate with: model='{PUSH_REPO}'")
